# FAIR Universe - Weak Lensing ML Uncertainty Challenge
## Phase 2 Starting Kit: Power Spectrum Analysis
***

In the universe, massive objects like galaxies and clusters of galaxies bend the path of light traveling near them — a phenomenon known as gravitational lensing, as predicted by Einstein’s general relativity. When this bending is subtle, it’s known as weak gravitational lensing. It slightly distorts the shapes of distant galaxies as their light travels through the cosmic web of matter, including dark matter and ordinary matter (baryons), on its way to Earth.

By measuring these tiny distortions across the sky, we can reconstruct convergence maps — essentially cosmic “heat maps” that show where matter is concentrated, even if that matter is invisible. These maps let us trace the hidden structure of the universe.

Crucially, weak lensing maps contain rich cosmological information. By statistically analyzing the patterns in these distortions, we can learn about the universe’s content and evolution. For example, we can estimate how much dark matter and dark energy exist, how fast the universe is expanding, and how structures have grown over time. In this way, weak lensing helps us constrain our cosmological model.

So far, the most widely accepted model of the universe is called $\Lambda$CDM (Lambda Cold Dark Matter). It describes a universe dominated by dark energy (represented by the Greek letter $\Lambda$, or Lambda) and cold dark matter, with only a small fraction of normal matter, such as stars and planets. Weak lensing is one of the most powerful tools in cosmology; however, various systematic effects and unknown physics can distort weak lensing maps and introduce uncertainties when estimating parameters of physics models from them. In addition, the real data observed from telescopes could also be different from the detailed setup of our simulation data that is used to train our models.

The goal of this competition is to identify the weak lensing convergence maps that were generated by unknown systematic effects. The test data is generated assuming different physical models (**out-of-distribution**; **OoD**), and participants' methods should give a continuous OoD score $t(x)$ that increases monotonically with the confidence that their models predict a given sample as OoD.

Participants will not be provided with OoD examples or any information on how the OoD test data are generated. The OoD score $t(x)$ could be, for example, any test statistic increasing monotonically with the OoD likelihood, or the $p$-value defined from the test statistics of the training data and test data. The participants' models will then be scored by a ROC-AUC metric we defined for this competition.

***

`COLAB` determines whether this notebook is running on Google Colab.

In [1]:
COLAB = 'google.colab' in str(get_ipython())

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
if COLAB:
    # clone github repo
    !git clone --depth 1 https://github.com/FAIR-Universe/Cosmology_Challenge.git
    # move to the HEP starting kit folder
    %cd Cosmology_Challenge/

Cloning into 'Cosmology_Challenge'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 87 (delta 10), reused 63 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 110.31 MiB | 15.35 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/Cosmology_Challenge


# 0 - Imports & Settings

In [11]:
class Utility:
    @staticmethod
    def add_noise(data, mask, ng, pixel_size=2.):
        """
        Add noise to a noiseless convergence map.

        Parameters
        ----------
        data : np.array
            Noiseless convergence maps.
        mask : np.array
            Binary mask map.
        ng : float
            Number of galaxies per arcmin². This determines the noise level; a larger number means smaller noise.
        pixel_size : float, optional
            Pixel size in arcminutes (default is 2.0).
        """

        return data + np.random.randn(*data.shape) * 0.4 / (2*ng*pixel_size**2)**0.5 * mask

    @staticmethod
    def load_np(data_dir, file_name):
        file_path = os.path.join(data_dir, file_name)
        return np.load(file_path)

    @staticmethod
    def save_np(data_dir, file_name, data):
        file_path = os.path.join(data_dir, file_name)
        np.save(file_path, data)

    @staticmethod
    def save_json_zip(submission_dir, json_file_name, zip_file_name, data):
        """
        Save a dictionary with 'means' and 'errorbars' into a JSON file,
        then compress it into a ZIP file inside submission_dir.

        Parameters
        ----------
        submission_dir : str
            Path to the directory where the ZIP file will be saved.
        file_name : str
            Name of the ZIP file (without extension).
        data : dict
            Dictionary with keys 'means' and 'errorbars'.

        Returns
        -------
        str
            Path to the created ZIP file.
        """
        os.makedirs(submission_dir, exist_ok=True)

        json_path = os.path.join(submission_dir, json_file_name)

        # Save JSON file
        with open(json_path, "w") as f:
            json.dump(data, f)

        # Path to ZIP
        zip_path = os.path.join(submission_dir, zip_file_name)

        # Create ZIP containing only the JSON
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            zf.write(json_path, arcname=json_file_name)

        # Remove the standalone JSON after zipping
        os.remove(json_path)

        return zip_path

In [4]:
import os
import json
import time
import zipfile
import datetime
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.metrics import roc_curve

import scipy
from scipy.interpolate import LinearNDInterpolator

# 1 - Helper Classes for
- Utitlity Functions
- Data Loading
- Visualizations
- Scoring Functions

### Utility

### Data

In [5]:
class Data:
    def __init__(self, data_dir, USE_PUBLIC_DATASET):
        self.USE_PUBLIC_DATASET = USE_PUBLIC_DATASET
        self.data_dir = data_dir
        self.mask_file = 'WIDE12H_bin2_2arcmin_mask.npy'
        self.viz_label_file = 'label_newrealization.npy'
        if self.USE_PUBLIC_DATASET:
            self.kappa_file = 'WIDE12H_bin2_2arcmin_kappa_newrealization.npy'
            self.label_file = self.viz_label_file
            self.Ncosmo = 101  # Number of cosmologies in the entire training data
            self.Nsys = 256    # Number of systematic realizations in the entire training data
            self.test_kappa_file = 'WIDE12H_bin2_2arcmin_kappa_test_phase2_new_v2.npy'
            self.Ntest = 10000  # Number of instances in the test data
        else:
            self.kappa_file = 'sampled_WIDE12H_bin2_2arcmin_kappa.npy'
            self.label_file = 'sampled_label.npy'
            self.Ncosmo = 3    # Number of cosmologies in the sampled training data
            self.Nsys = 30     # Number of systematic realizations in the sampled training data
            self.test_kappa_file = 'sampled_WIDE12H_bin2_2arcmin_kappa_noisy_test.npy'
            self.Ntest = 3     # Number of instances in the sampled test data

        self.shape = [1424,176] # dimensions of each map
        self.pixelsize_arcmin = 2 # pixel size in arcmin
        self.pixelsize_radian = self.pixelsize_arcmin / 60 / 180 * np.pi # pixel size in radian
        self.ng = 30  # galaxy number density. This determines the noise level of the experiment. Do not change this number.

    def load_train_data(self):
        self.mask = Utility.load_np(data_dir=self.data_dir, file_name=self.mask_file) # A binary map that shows which parts of the sky are observed and which areas are blocked
        self.kappa = np.zeros((self.Ncosmo, self.Nsys, *self.shape), dtype=np.float16)
        self.kappa[:,:,self.mask] = Utility.load_np(data_dir=self.data_dir, file_name=self.kappa_file) # Training convergence maps
        self.label = Utility.load_np(data_dir=self.data_dir, file_name=self.label_file) # Training labels (cosmological and physical paramameters) of each training map
        self.viz_label = Utility.load_np(data_dir=self.data_dir, file_name=self.viz_label_file) # For visualization of parameter distributions

    def load_test_data(self):
        self.kappa_test = np.zeros((self.Ntest, *self.shape), dtype=np.float16)
        self.kappa_test[:,self.mask] = Utility.load_np(data_dir=self.data_dir, file_name=self.test_kappa_file) # Test noisy convergence maps

### Visualization

# 2 - Load train and test data

The training maps are generated by $N_{\rm cosmo}$ cosmological models, each model contains $N_{\rm sys}$ realizations with different nuisance parameters. So the shape of the training maps kappa is $(N_{\rm cosmo}, N_{\rm sys}, 1424, 176)$.

Each training map is associated with 5 parameters, so the shape of label is $(N_{\rm cosmo}, N_{\rm sys}, 5)$. The first two parameters are cosmological parameters $\Omega_m$ and $S_8$, while the rest three parameters are nuisance parameters that describe systematic effects and need to be marginalized in the data analysis (two of them describe baryonic effects and the last one describes photometric redshift uncertainties).

In [12]:
root_dir = os.getcwd()
print("Root directory is", root_dir)

Root directory is /content/Cosmology_Challenge


**Option 1: To quickly run through this starting kit:** You may set `USE_PUBLIC_DATASET = False` so that only a downsampled training data and test data will be loaded. In the downsampled training data, there are $N_{\rm cosmo}=3$ cosmological models and $N_{\rm sys}=30$ realizations of nuisance parameters. The downsampled test data contains $N_{\rm test}=3$ instances.

***

#### ⚠️ NOTE:
To make a valid submission and obtain a score on Codabench, **you will need to make your predictions using the entire test data set (10,000 instances). Use the Option 2 below to load the entire test data.**

***

**Option 2: To load the entire training data and test data:** Set `USE_PUBLIC_DATASET = True` and specify a path where you will save the downloaded public data from Codabench. In the entire training data, there are $N_{\rm cosmo}=101$ cosmological models and $N_{\rm sys}=256$ realizations of nuisance parameters. The entire test data contains $N_{\rm test}=10000$ instances.

In [13]:
# USE_PUBLIC_DATASET = False

USE_PUBLIC_DATASET = True
PUBLIC_DATA_DIR = os.path.join(root_dir, '../drive/MyDrive/cosmology_challenge_data/') # '[DEFINE THE PATH OF SAVED PUBLIC DATA HERE]'  # This is only required when you set USE_PUBLIC_DATASET = True

In [14]:
if not USE_PUBLIC_DATASET:                                         # Testing this startking kit with a tiny sample of the training data (3, 30, 1424, 176)
    DATA_DIR = os.path.join(root_dir, 'input_data/')
else:                                                              # Training your model with all training data (101, 256, 1424, 176)
    DATA_DIR = PUBLIC_DATA_DIR

### Load the train and test data

In [15]:
# Initialize Data class object
data_obj = Data(data_dir=DATA_DIR, USE_PUBLIC_DATASET=USE_PUBLIC_DATASET)

# Load train data
data_obj.load_train_data()

# Load test data
data_obj.load_test_data()

#### ⚠️ NOTE:
- The original training images are *noiseless* (without any pixel-level noise).
- The original test images is *noisy* (pixel-level noise with galaxy number density $n_g = 30~\text{arcmin}^{-2}$ and pixel size $=2$ arcmin has been added).

- **You will have to add pixel-level noise to the training data through the helper function** `Utility.add_noise`.
    
For example:
```python
    noisy_kappa = Utility.add_noise(data=data_obj.kappa.astype(np.float64),
                                    mask=data_obj.mask,
                                    ng=data_obj.ng,
                                    pixel_size=data_obj.pixelsize_arcmin)
```

The shape of `noisy_kappa` will be the same as the shape of `data_obj.kappa`.

In [16]:
Ncosmo = data_obj.Ncosmo
Nsys = data_obj.Nsys

print(f'There are {Ncosmo} cosmological models, each has {Nsys} realizations of nuisance parameters in the training data.')

There are 101 cosmological models, each has 256 realizations of nuisance parameters in the training data.


In [14]:
print(f'Shape of the training data = {data_obj.kappa.shape}')
print(f'Shape of the mask = {data_obj.mask.shape}')
print(f'Shape of the training label = {data_obj.label.shape}')
print(f'Shape of the test data = {data_obj.kappa_test.shape}')

Shape of the training data = (101, 256, 1424, 176)
Shape of the mask = (1424, 176)
Shape of the training label = (101, 256, 5)
Shape of the test data = (10000, 1424, 176)


# 4 - Training (build power spectrum and covariance emulator)

In cosmology, the power spectrum describes how matter is distributed across different size scales in the universe and is a key tool for studying the growth of cosmic structure. Starting from the matter density $\delta(x)$, we transform it into Fourier space to get $\tilde{\delta}(x)$, which represents fluctuations as waves of different wavelengths. The matter power spectrum P(k) is then defined by:

\begin{equation}
\langle \tilde{\delta}(\mathbf{k}) \tilde{\delta}^*(\mathbf{k}') \rangle = (2\pi)^3 \delta_D(\mathbf{k}-\mathbf{k}') P(k),
\end{equation}

where k is the wavenumber corresponding to a scale $\lambda \sim 1/k$, and $ \delta_D$ is the Dirac delta function. Intuitively, P(k) tells us how "clumpy" the universe is on different scales. In cosmology, the shape and amplitude of P(k) encodes the physics and composition of the universe, making it one of the most important statistical tools in the field.

In this notebook we use power spectrum as the summary statistic to constrain the cosmological parameters.

In [6]:
Nbins = 10
l_edge = np.logspace(2, 4, Nbins+1)

In [7]:
def power_spectrum(x, pixsize, kedge):
    """
    Compute the azimuthally averaged 2D power spectrum of a real-valued 2D field.

    Parameters:
    -----------
    x : 2D numpy array
        Input real-space map (e.g., an image or simulated field).
        Must be a 2D array with shape (N_y, N_x).

    pixsize : float
        Physical size of each pixel in the map (e.g., arcmin, Mpc, etc.).
        Units should be consistent with the units used for `kedge`.

    kedge : 1D array-like
        Bin edges in wavenumber space (k), used to bin the power spectrum.
        Should be monotonically increasing and cover the k-range of interest.

    Returns:
    --------
    power_k : 1D numpy array
        The average wavenumber in each k bin (excluding the DC bin).

    power : 1D numpy array
        The binned, azimuthally averaged power spectrum corresponding to `power_k`.
        Normalized per unit area.
    """

    # Ensure the input array is 2D
    assert x.ndim == 2

    # Compute the 2D FFT of the input map and take its squared magnitude (power spectrum)
    xk = np.fft.rfft2(x)  # Real-to-complex FFT (along last axis)
    xk2 = (xk * xk.conj()).real  # Power spectrum: |FFT|^2

    # Get the shape of the input map
    Nmesh = x.shape

    # Compute the wavenumber grid (k-space)
    k = np.zeros((Nmesh[0], Nmesh[1]//2+1))
    # Square of the frequency in the first axis
    k += np.fft.fftfreq(Nmesh[0], d=pixsize).reshape(-1, 1) ** 2
    # Square of the frequency in the second axis (real FFT)
    k += np.fft.rfftfreq(Nmesh[1], d=pixsize).reshape(1, -1) ** 2
    # Convert from (1/length)^2 to angular frequency in radian units
    k = k ** 0.5 * 2 * np.pi

    # Bin each k value according to the bin edges provided in kedge
    index = np.searchsorted(kedge, k)

    # Bin the power values, number of modes, and wavenumbers
    power = np.bincount(index.flatten(), weights=xk2.flatten())
    Nmode = np.bincount(index.flatten())
    power_k = np.bincount(index.flatten(), weights=k.flatten())

    # Adjust for symmetry in the real FFT: include the mirrored part (excluding Nyquist frequency)
    if Nmesh[1] % 2 == 0:  # Even number of columns
        power += np.bincount(index[...,1:-1].flatten(), weights=xk2[...,1:-1].flatten())
        Nmode += np.bincount(index[...,1:-1].flatten())
        power_k += np.bincount(index[...,1:-1].flatten(), weights=k[...,1:-1].flatten())
    else:  # Odd number of columns
        power += np.bincount(index[...,1:].flatten(), weights=xk2[...,1:].flatten())
        Nmode += np.bincount(index[...,1:].flatten())
        power_k += np.bincount(index[...,1:].flatten(), weights=k[...,1:].flatten())

    # Exclude the first bin (typically corresponds to DC mode)
    power = power[1:len(kedge)]
    Nmode = Nmode[1:len(kedge)]
    power_k = power_k[1:len(kedge)]

    # Average the power and wavenumber in each bin, only where Nmode > 0
    select = Nmode > 0
    power[select] = power[select] / Nmode[select]
    power_k[select] = power_k[select] / Nmode[select]

    # Normalize the power spectrum by the map area
    power *= pixsize ** 2 / Nmesh[0] / Nmesh[1]

    # Return the binned k values and corresponding power spectrum
    return power_k, power

## Advanced Strategy: Multi-Summary Statistics + Normalizing Flows

We will extract **Peak Counts** from the weak lensing convergence maps and combine them with the **Power Spectrum**. Then, we will train a **Normalizing Flow** (specifically a Masked Autoregressive Flow) to accurately estimate the joint density $p(\mathbf{d})$. Out-of-distribution (OoD) maps will be identified by having extremely low probability (high negative log-likelihood) under this flow.

In [8]:
!pip install -q nflows

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [17]:
import torch
# import joblib
from torch import nn
from torch import optim
from scipy.ndimage import maximum_filter
from nflows.flows import Flow
from nflows.distributions import StandardNormal
from nflows.transforms import CompositeTransform, InverseTransform, RandomPermutation
from nflows.transforms.autoregressive import MaskedAffineAutoregressiveTransform
from sklearn.preprocessing import StandardScaler

def extract_peak_counts(map_2d, thresholds=np.linspace(-0.02, 0.06, 8)):
    """Extracts the local peak counts above different threshold levels."""
    local_max = (maximum_filter(map_2d, size=3) == map_2d)
    peaks = map_2d[local_max]
    counts, _ = np.histogram(peaks, bins=thresholds)
    return counts

# 1. Compute multi-summary statistics (Power Spectrum + Peak Counts) for training data
print("Extracting joint summary statistics for training data...")
num_peaks_features = 7  # 8 bins edge -> 7 features
joint_features = np.zeros((Ncosmo, Nsys, Nbins + num_peaks_features))

for i in range(Ncosmo):
    for j in range(Nsys):
        noisy_map = Utility.add_noise(
            data=data_obj.kappa[i,j].astype(np.float64),
            mask=data_obj.mask,
            ng=data_obj.ng,
            pixel_size=data_obj.pixelsize_arcmin
        )
        # Power spectrum
        _, ps = power_spectrum(noisy_map, data_obj.pixelsize_radian, l_edge)
        # Peak counts
        pc = extract_peak_counts(noisy_map)
        # Concatenate features
        joint_features[i, j] = np.concatenate([np.log10(ps), pc])

joint_features_flat = joint_features.reshape(-1, Nbins + num_peaks_features)

# Standardize joint summary statistics
scaler_joint = StandardScaler()
joint_features_scaled = scaler_joint.fit_transform(joint_features_flat)
print("Features extracted and standardized. Shape:", joint_features_scaled.shape)
print(type(joint_features_scaled))
# joblib.dump(joint_features_scaled, 'scaler.pkl')

Extracting joint summary statistics for training data...
Features extracted and standardized. Shape: (25856, 17)
<class 'numpy.ndarray'>


In [22]:
# 2. Define and train Masked Autoregressive Flow (MAF)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_features = Nbins + num_peaks_features

num_layers = 5
base_dist = StandardNormal(shape=[num_features])
transforms = []
for _ in range(num_layers):
    transforms.append(MaskedAffineAutoregressiveTransform(features=num_features,
                                                          hidden_features=64,
                                                          context_features=None))
    transforms.append(RandomPermutation(features=num_features))

transform = CompositeTransform(transforms)
flow = Flow(transform, base_dist).to(device)
optimizer = optim.Adam(flow.parameters(), lr=1e-3, weight_decay=1e-5)

# Prepare PyTorch DataLoader
x_train_tensor = torch.tensor(joint_features_scaled, dtype=torch.float32).to(device)
train_dataset = torch.utils.data.TensorDataset(x_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)

print(f"Training Normalizing Flow on {device}...")
flow.train()
for epoch in range(150):
    loss_sum = 0.0
    for batch in train_loader:
        x_batch = batch[0]
        optimizer.zero_grad()
        loss = -flow.log_prob(inputs=x_batch).mean()
        loss.backward()
        optimizer.step()
        loss_sum += loss.item()
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1}/150 - Mean Neg Log-Likelihood: {loss_sum/len(train_loader):.4f}")

Training Normalizing Flow on cpu...
Epoch 30/150 - Mean Neg Log-Likelihood: 11.7813
Epoch 60/150 - Mean Neg Log-Likelihood: 11.7015
Epoch 90/150 - Mean Neg Log-Likelihood: 11.6124
Epoch 120/150 - Mean Neg Log-Likelihood: 11.5137
Epoch 150/150 - Mean Neg Log-Likelihood: 11.4148


In [21]:
# 3. Extract same joint features on Test Data
print("Extracting joint summary statistics for test data...")
test_joint_features = np.zeros((data_obj.Ntest, Nbins + num_peaks_features))

for i in range(data_obj.Ntest):
    # Note: Test data already contains noise
    test_map = data_obj.kappa_test[i].astype(np.float64)
    _, ps = power_spectrum(test_map, data_obj.pixelsize_radian, l_edge)
    pc = extract_peak_counts(test_map)
    test_joint_features[i] = np.concatenate([np.log10(ps), pc])

test_joint_scaled = scaler_joint.transform(test_joint_features)
test_tensor = torch.tensor(test_joint_scaled, dtype=torch.float32).to(device)

# 4. Evaluate OoD Scores via Negative Log-Likelihood from the flow
print("Evaluating OoD scores on test set...")
flow.eval()
with torch.no_grad():
    # Evaluate log probabilities in batches to avoid GPU OOM
    log_probs = []
    test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(test_tensor), batch_size=1000, shuffle=False)
    for batch in test_loader:
        x_batch = batch[0]
        log_probs.append(flow.log_prob(x_batch).cpu().numpy())
    log_probs = np.concatenate(log_probs)

# Higher negative log-probability = more likely OoD
ood_scores = -log_probs
print("OoD scores calculated! Ready to package and submit.")

Extracting joint summary statistics for test data...
Evaluating OoD scores on test set...
OoD scores calculated! Ready to package and submit.


#### ⚠️ NOTE:
In this starting kit example, you may also directly submit $\chi^2_{\rm test}$ (the test statistics of the test samples) as the OoD scores. However, keep in mind that for other methods, the distribution of the test statistics could be two-sided, which means that OoD samples could be detected by either small or large test statistics. To obtain the best ROC-AUC score, we recommend defining an OoD score that increases monotonically with the OoD probability.

# 6 - Prepare submission for Codabench

***

This section will save the model predictions `ood_scores` (an array with shape `(10000,)`, where `10000` is the number of test instances) as a dictionary in a JSON file `result.json`. Then it will compress `result.json` into a zip file that can be directly submitted to Codabench.

***

In [ ]:
data = {"ood_scores": ood_scores.tolist()}
the_date = datetime.datetime.now().strftime("%y-%m-%d-%H-%M")
zip_file_name = 'Submission_' + the_date + '.zip'
zip_file = Utility.save_json_zip(
    submission_dir="submissions",
    json_file_name="result.json",
    zip_file_name=zip_file_name,
    data=data
)
print(f"Submission ZIP saved at: {zip_file}")

Submission ZIP saved at: submissions/Submission_26-10-07-14-20.zip
